In [23]:
from datetime import timedelta
import pandas as pd
import numpy as np

# Load the outage data
outage_df = pd.read_csv("western_interconnection_outages_fips_filtered.csv")
outage_df['start_time'] = pd.to_datetime(outage_df['start_time'])
outage_df['end_time'] = pd.to_datetime(outage_df['end_time'])



In [ ]:

# Define the exclusion buffer and number of negatives per event
exclusion_buffer = timedelta(days=2)
negatives_per_event = 2

# Precompute candidate hours for 10 years
all_hours = pd.date_range(start='2014-01-01', end='2023-12-31 23:00:00', freq='H')

# Group by county
grouped = outage_df.groupby("fips")

# List to hold negatives
negative_samples = []

# Loop through each outage
for idx, row in outage_df.iterrows():
    fips = row["fips"]
    state = row["state"]
    county = row["county"]
    event_time = row["start_time"]
    
    # Get all outages for this county
    county_outages = grouped.get_group(fips)
    
    # Build exclusion window for all outages in that county
    exclusion_hours = set()
    for _, event in county_outages.iterrows():
        exclusion_start = event["start_time"] - exclusion_buffer
        exclusion_end = event["end_time"] + exclusion_buffer
        exclusion_hours.update(pd.date_range(start=exclusion_start, end=exclusion_end, freq='h'))
    
    # Candidate pool: hours from same county, excluding buffer
    valid_hours = pd.Series([h for h in all_hours if h not in exclusion_hours])
    
    # Sample negatives
    if len(valid_hours) >= negatives_per_event:
        sampled = valid_hours.sample(n=negatives_per_event, random_state=idx)  # seed with row index
        for ts in sampled:
            negative_samples.append({
                "fips": fips,
                "state": state,
                "county": county,
                "start_time": ts,
                "duration": 0.0,
                "label": 0
            })

# Convert to DataFrame
negative_df = pd.DataFrame(negative_samples)
negative_df.to_csv("negatives_per_outage.csv", index=False)

print(f"Generated {len(negative_df)} negatives.")
negative_df.head()

Generated 2010 negative samples.


,fips,state,county,start_time,duration,label
0,4003,Arizona,Cochise,2015-09-06 05:00:00,0.0,0
1,4003,Arizona,Cochise,2017-03-25 07:00:00,0.0,0
2,4003,Arizona,Cochise,2018-07-06 09:00:00,0.0,0
3,4003,Arizona,Cochise,2023-11-23 06:00:00,0.0,0
4,4003,Arizona,Cochise,2018-09-27 21:00:00,0.0,0
